# SAIA2163 NLP Final Project — Multilingual Improved Version

This notebook changes the project from English-only sentiment analysis to multilingual sentiment analysis using all available languages in the raw dataset:

- EN: English
- FR: French
- DE: German
- JP: Japanese
- IT: Italian

Main change: this version **does not use English-only lemmatization or English-only stopword removal**. Instead, it uses Unicode-safe light cleaning, language tokens, word n-grams, character n-grams, class balancing, Macro-F1, neutral recall, and per-language evaluation.


In [1]:
from pathlib import Path

DATA_FILE = "Training_Data_Google_Play_reviews_6000.csv"
DATA_PATH = Path.cwd() / DATA_FILE

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}"
        f"Place {DATA_FILE} in the same folder as this notebook."
    )

print("Using dataset:", DATA_PATH.resolve())


Using dataset: C:\Coding space\Assignment nlp\NLP\SAIA2163-NLP-Final-Project\Training_Data_Google_Play_reviews_6000.csv


## 1. Import multilingual utilities and training functions

The notebook uses the same code as the Python script so the notebook, training script, and Streamlit app stay consistent.


In [2]:
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from preprocessing_utils import LABEL_NAMES, INVERSE_LABEL_MAPPING
from train_multilingual_models import (
    RANDOM_STATE,
    load_and_prepare_dataset,
    build_multilingual_models,
    evaluate_pipeline,
    language_level_metrics,
    plot_confusion_matrix,
    save_visuals,
)


## 2. Load and prepare the full multilingual dataset

This step:

1. Keeps all supported languages instead of filtering English only.
2. Normalizes `userLang`, for example `" EN"` becomes `"EN"`.
3. Maps scores to sentiment labels.
4. Creates `model_input` with language tokens such as `lang_en`, `lang_fr`, `lang_de`, `lang_jp`, and `lang_it`.
5. Saves `preprocessed_reviews.csv`.


In [3]:
df = load_and_prepare_dataset(DATA_PATH, Path.cwd())

print("Rows after preprocessing:", len(df))
print("Language distribution:")
print(df["userLang"].value_counts().sort_index())
print("Sentiment distribution:")
print(df["label"].value_counts().reindex(LABEL_NAMES).fillna(0).astype(int))
print("Sentiment by language:")
print(pd.crosstab(df["userLang"], df["label"]).reindex(columns=LABEL_NAMES).fillna(0).astype(int))

df[["userLang", "raw_text", "model_input", "text", "label"]].head(10)


Rows after preprocessing: 6000
Language distribution:
userLang
DE    1200
EN    1200
FR    1200
IT    1200
JP    1200
Name: count, dtype: int64
Sentiment distribution:
label
negative    2302
neutral      453
positive    3245
Name: count, dtype: int64
Sentiment by language:
label     negative  neutral  positive
userLang                             
DE             646      106       448
EN             444       70       686
FR             302      103       795
IT             466      104       630
JP             444       70       686


,userLang,raw_text,model_input,text,label
0,EN,Gett van for no reason 😂😂😂,lang_en Gett van for no reason 😂😂😂,lang_en gett van for no reason 😂😂😂,negative
1,EN,better' than WhatsApp,lang_en better' than WhatsApp,lang_en better' than whatsapp,positive
2,EN,That was good app for me,lang_en That was good app for me,lang_en that was good app for me,positive
3,EN,Love the app,lang_en Love the app,lang_en love the app,positive
4,EN,🕳️🕳️🕳️,lang_en 🕳️🕳️🕳️,lang_en 🕳️🕳️🕳️,negative
5,EN,Good app,lang_en Good app,lang_en good app,positive
6,EN,Nice app,lang_en Nice app,lang_en nice app,positive
7,EN,telegram mare call par nai ban rhi,lang_en telegram mare call par nai ban rhi,lang_en telegram mare call par nai ban rhi,positive
8,EN,now how to mute stories?,lang_en now how to mute stories?,lang_en now how to mute stories?,neutral
9,EN,Good parfom sab 1,lang_en Good parfom sab 1,lang_en good parfom sab 1,positive


## 3. Train/test split

The split is stratified by sentiment label so each class is represented in train and test sets.


In [4]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)

X_train = train_df["model_input"]
X_test = test_df["model_input"]
y_train = train_df["target"]
y_test = test_df["target"]

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))


Train rows: 4800
Test rows: 1200


## 4. Train multilingual traditional ML models

Models tested:

- Multilingual Logistic Regression with word TF-IDF
- Multilingual Naive Bayes with word TF-IDF
- Multilingual Logistic Regression with character n-grams
- Multilingual Logistic Regression with combined word + character TF-IDF

Character n-grams are useful because the data contains multiple languages, misspellings, short reviews, and Japanese text.


In [5]:
results = []

for name, model in build_multilingual_models().items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    results.append(evaluate_pipeline(name, model, X_test, y_test))


Training Multilingual Logistic Regression Word TF-IDF...

==== Multilingual Logistic Regression Word TF-IDF ====
        accuracy: 0.7633
       precision: 0.7473
          recall: 0.7633
        f1_score: 0.7529
 macro_precision: 0.6174
    macro_recall: 0.5901
        macro_f1: 0.5951
  neutral_recall: 0.1538
              precision    recall  f1-score   support

    negative       0.72      0.78      0.75       460
     neutral       0.30      0.15      0.20        91
    positive       0.83      0.84      0.83       649

    accuracy                           0.76      1200
   macro avg       0.62      0.59      0.60      1200
weighted avg       0.75      0.76      0.75      1200

Training Multilingual Naive Bayes Word TF-IDF...

==== Multilingual Naive Bayes Word TF-IDF ====
        accuracy: 0.7783
       precision: 0.7938
          recall: 0.7783
        f1_score: 0.7471
 macro_precision: 0.8495
    macro_recall: 0.5550
        macro_f1: 0.5396
  neutral_recall: 0.0110
         

## 5. Compare models

For this project, **Macro-F1** and **neutral recall** are important because the neutral class is smaller than positive and negative.


In [6]:
results_df = pd.DataFrame([
    {k: v for k, v in result.items() if k not in {"pipeline", "predictions", "confusion_matrix", "classification_report"}}
    for result in results
])

results_df = results_df.sort_values(["macro_f1", "f1_score", "accuracy"], ascending=False).reset_index(drop=True)
results_df.to_csv("model_evaluation_comparison.csv", index=False)
results_df


,model_name,accuracy,precision,recall,f1_score,macro_precision,macro_recall,macro_f1,neutral_recall
0,Multilingual SGD Character N-grams,0.796667,0.785568,0.796667,0.790121,0.666567,0.640684,0.649796,0.252747
1,Multilingual SGD Word + Character TF-IDF,0.802500,0.787405,0.802500,0.791676,0.684854,0.633370,0.645934,0.208791
2,Multilingual Logistic Regression Word TF-IDF,0.763333,0.747277,0.763333,0.752947,0.617439,0.590107,0.595115,0.153846
3,Multilingual Naive Bayes Word TF-IDF,0.778333,0.793837,0.778333,0.747063,0.849474,0.555033,0.539554,0.010989


## 6. Save the best model and backend files

The best model is selected using Macro-F1 first, then weighted F1-score and accuracy.


In [7]:
best_name = results_df.loc[0, "model_name"]
best_result = next(result for result in results if result["model_name"] == best_name)
best_pipeline = best_result["pipeline"]
best_predictions = best_result["predictions"]

print("Best model:", best_name)
joblib.dump(best_pipeline, "best_sentiment_pipeline.pkl")

predictions_df = test_df[["raw_text", "model_input", "text", "label", "target", "userLang", "language_name", "app_name", "review_date"]].copy()
predictions_df["predicted_target"] = best_predictions
predictions_df["predicted_label"] = predictions_df["predicted_target"].map(INVERSE_LABEL_MAPPING)
predictions_df.to_csv("final_model_predictions.csv", index=False)

language_metrics_df = language_level_metrics(test_df, best_predictions, Path.cwd())
language_metrics_df


Best model: Multilingual SGD Character N-grams


,language,language_name,rows,accuracy,precision,recall,f1_score,macro_precision,macro_recall,macro_f1,neutral_recall
0,DE,German,238,0.781513,0.783853,0.781513,0.780614,0.555448,0.550000,0.551248,0.000000
1,EN,English,237,0.898734,0.896679,0.898734,0.897339,0.836739,0.808863,0.821595,0.600000
2,FR,French,231,0.666667,0.650179,0.666667,0.657713,0.485895,0.479956,0.481110,0.130435
3,IT,Italian,254,0.744094,0.710816,0.744094,0.724219,0.571678,0.565858,0.561316,0.080000
4,JP,Japanese,240,0.891667,0.891807,0.891667,0.889331,0.894065,0.802756,0.838065,0.600000


## 7. Save metadata and visualizations


In [8]:
from preprocessing_utils import LABEL_MAPPING, SUPPORTED_LANGUAGES

metadata = {
    "project_type": "multilingual_sentiment_analysis",
    "best_model_name": str(best_name),
    "label_mapping": LABEL_MAPPING,
    "inverse_label_mapping": INVERSE_LABEL_MAPPING,
    "label_names": LABEL_NAMES,
    "supported_languages": SUPPORTED_LANGUAGES,
    "uses_language_token": True,
    "preprocessing": "Unicode-safe multilingual cleaning; no English-only lemmatization or stopword removal.",
    "dataset_rows_after_preprocessing": int(len(df)),
    "language_distribution": df["userLang"].value_counts().sort_index().to_dict(),
    "sentiment_distribution": df["label"].value_counts().to_dict(),
    "best_metrics": {k: float(best_result[k]) for k in ["accuracy", "precision", "recall", "f1_score", "macro_f1", "neutral_recall"]},
}

joblib.dump(metadata, "backend_metadata.pkl")

import json
with open("backend_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

Path("images").mkdir(exist_ok=True)
plot_confusion_matrix(best_result["confusion_matrix"], f"Confusion Matrix - {best_name}", Path("images") / "confusion_matrix_best_model.png")
save_visuals(df, results_df, Path.cwd())

print("Saved model, metadata, CSV files, and images.")


Saved model, metadata, CSV files, and images.


## 8. Optional GridSearchCV

This is slower. Run it only if you have time. It may improve Macro-F1.


In [9]:
# Optional. Uncomment to run slower GridSearchCV tuning.
from train_multilingual_models import run_grid_search
grid_model = run_grid_search(X_train, y_train)
grid_result = evaluate_pipeline("GridSearch Multilingual Word + Character TF-IDF", grid_model, X_test, y_test)


Fitting 3 folds for each of 48 candidates, totalling 144 fits

Best grid parameters:
{
  "classifier__estimator__C": 2.0,
  "features__char__max_features": 12000,
  "features__char__ngram_range": [
    2,
    4
  ],
  "features__word__max_features": 15000,
  "features__word__ngram_range": [
    1,
    2
  ]
}
Best CV macro-F1: 0.6232

==== GridSearch Multilingual Word + Character TF-IDF ====
        accuracy: 0.8017
       precision: 0.7868
          recall: 0.8017
        f1_score: 0.7914
 macro_precision: 0.6779
    macro_recall: 0.6322
        macro_f1: 0.6436
  neutral_recall: 0.2088
              precision    recall  f1-score   support

    negative       0.77      0.81      0.79       460
     neutral       0.41      0.21      0.28        91
    positive       0.85      0.88      0.87       649

    accuracy                           0.80      1200
   macro avg       0.68      0.63      0.64      1200
weighted avg       0.79      0.80      0.79      1200



## 9. Optional multilingual Transformer

For multilingual data, use `xlm-roberta-base` instead of English DistilBERT. This is slower and works best with GPU.


In [10]:
# Optional. Uncomment to run multilingual transformer training.
from train_multilingual_models import train_optional_transformer
transformer_result = train_optional_transformer(train_df, test_df, Path.cwd())


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
c:\Coding space\Assignment nlp\NLP\SAIA2163-NLP-Final-Project\.venv\Lib\site-packages\torch\ut

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Neutral Recall
1,0.604751,0.625046,0.795833,0.545639,0.000000
2,0.686626,0.554921,0.813333,0.559212,0.000000
3,0.488372,0.564041,0.819167,0.563593,0.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Coding space\Assignment nlp\NLP\SAIA2163-NLP-Final-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Coding space\Assignment nlp\NLP\SAIA2163-NLP-Final-Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Suggested report explanation

> The multilingual improvement uses the full dataset across English, French, German, Japanese, and Italian reviews. Since English-only lemmatization and stopword removal are not suitable for multilingual text, preprocessing was changed to Unicode-safe light cleaning. Language tokens were added to help traditional ML models learn language-specific patterns. Word-level and character-level TF-IDF n-gram models were evaluated using accuracy, weighted F1-score, macro-F1, neutral recall, and per-language metrics.
